In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# ======================================================
# Style
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

sns.set_style("white")

# ======================================================
# Config
# ======================================================

OUTDIR = "figs2_final_validation"
os.makedirs(OUTDIR, exist_ok=True)

LR = "0.0001"
BS = "32"

BAR_WIDTH = 0.25
GROUP_SPACING = 0.82

MODEL_COLORS = {
    "MLP": "#6EB8D2",
    "PCA": "#435385",
    "Transformer": "#8B1E1E"
}

DISEASE_ORDER = ["CHD", "stroke", "ICVD"]

DISEASE_XLABELS = [
    "Coronary\nHeart Disease",
    "Ischemic\nStroke",
    "ASCVD"
]

MODEL_ORDER = ["MLP", "PCA", "Transformer"]

# ======================================================
# Curve directories
# ======================================================

CURVE_DIRS = {
    "SNP": {
        "PCA": "code_02_02_save/SingleModal_Run_update_v5/snp_20251125_2051/curves",
        "MLP": "code_02_02_save/SingleModal_Run_update_v5/snp_20251125_2051/curves",
        "Transformer": "code_02_02_save/SingleModal_Run_update_v5/snp_20251125_2051/curves",
    },
    "Proteomics": {
        "PCA": "code_02_02_save/SingleModal_Run_update_v5/soma_20251123_1305/curves",
        "MLP": "code_02_02_save/SingleModal_Run_update_v5/soma_20251123_1306/curves",
        "Transformer": "code_02_02_save/SingleModal_Run_update_v5/soma_20251123_1306/curves",
    }
}

MODALITY_TAG = {
    "SNP": "snp",
    "Proteomics": "soma"
}

MODEL_TAG = {
    "PCA": "PCA",
    "MLP": "MLP",
    "Transformer": "TR"
}

# ======================================================
# Find best validation C-index
# ======================================================

def extract_fold(filename):
    m = re.search(r"_fold(\d+)_", filename)
    if m:
        return int(m.group(1))
    return np.nan


def get_best_valid_ci(modality, model, disease):
    curve_dir = CURVE_DIRS[modality][model]
    modality_tag = MODALITY_TAG[modality]
    model_tag = MODEL_TAG[model]

    pattern = os.path.join(
        curve_dir,
        f"{disease}_{modality_tag}_{model_tag}_fold*_lr{LR}_bs{BS}_train_curve.csv"
    )

    files = sorted(glob.glob(pattern))

    if len(files) == 0:
        print(f"[Missing] {modality} | {model} | {disease}")
        print("Pattern:", pattern)
        return None

    rows = []

    for f in files:
        try:
            df = pd.read_csv(f)
        except Exception as e:
            print(f"[Read error] {f}: {e}")
            continue

        if "valid_ci" not in df.columns:
            print(f"[No valid_ci] {f}")
            continue

        df["valid_ci"] = pd.to_numeric(df["valid_ci"], errors="coerce")
        df = df.dropna(subset=["valid_ci"])

        if len(df) == 0:
            continue

        idx = df["valid_ci"].idxmax()
        best = df.loc[idx]

        rows.append({
            "modality": modality,
            "model": model,
            "disease": disease,
            "fold": extract_fold(os.path.basename(f)),
            "best_epoch": int(best["epoch"]) if "epoch" in df.columns else np.nan,
            "best_valid_ci": float(best["valid_ci"]),
            "train_ci_at_best": float(best["train_ci"]) if "train_ci" in df.columns else np.nan,
            "train_loss_at_best": float(best["train_loss"]) if "train_loss" in df.columns else np.nan,
            "curve_file": f
        })

    if len(rows) == 0:
        return None

    tmp = pd.DataFrame(rows)
    best_row = tmp.loc[tmp["best_valid_ci"].idxmax()].copy()

    return best_row


all_results = []

for modality in ["SNP", "Proteomics"]:
    for model in MODEL_ORDER:
        for disease in DISEASE_ORDER:
            best_row = get_best_valid_ci(
                modality=modality,
                model=model,
                disease=disease
            )

            if best_row is not None:
                all_results.append(best_row)

results = pd.DataFrame(all_results)

summary_csv = os.path.join(
    OUTDIR,
    f"best_validation_valid_ci_lr{LR}_bs{BS}.csv"
)

results.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)
print(results)

# ======================================================
# Plot
# ======================================================

def add_value_labels(ax, bars):
    for bar in bars:
        h = bar.get_height()
        if np.isnan(h):
            continue
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.01,
            f"{h:.3f}",
            ha="center",
            va="bottom",
            fontsize=8
        )


def make_validation_barplot(modality):
    sub = results[results["modality"] == modality].copy()

    fig, ax = plt.subplots(figsize=(4.8, 4.0))

    x = np.arange(len(DISEASE_ORDER)) * GROUP_SPACING
    width = BAR_WIDTH

    for i, model in enumerate(MODEL_ORDER):
        vals = []

        for disease in DISEASE_ORDER:
            row = sub[
                (sub["disease"] == disease)
                & (sub["model"] == model)
            ]

            if len(row) == 0:
                vals.append(np.nan)
            else:
                vals.append(row.iloc[0]["best_valid_ci"])

        bars = ax.bar(
            x + (i - 1) * width,
            vals,
            width,
            color=MODEL_COLORS[model],
            edgecolor="black",
            linewidth=0.5,
            label=model,
            zorder=3
        )

        add_value_labels(ax, bars)

    ax.set_xticks(x)
    ax.set_xticklabels(DISEASE_XLABELS, fontsize=9)
    ax.set_ylabel("Validation C-index")
    ax.set_title(modality, fontsize=14)

    ax.set_ylim(0.45, 0.90)

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3,
        zorder=0
    )

    ax.legend(
        frameon=False,
        fontsize=10,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.12),
        ncol=3
    )

    sns.despine(ax=ax)

    plt.tight_layout()

    out_pdf = os.path.join(
        OUTDIR,
        f"{modality}_validation_valid_ci_barplot_lr{LR}_bs{BS}.pdf"
    )

    out_png = os.path.join(
        OUTDIR,
        f"{modality}_validation_valid_ci_barplot_lr{LR}_bs{BS}.png"
    )

    plt.savefig(out_pdf, dpi=300, transparent=True, bbox_inches="tight")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    print("Saved:", out_pdf)
    print("Saved:", out_png)


for modality in ["SNP", "Proteomics"]:
    make_validation_barplot(modality)

print("\nDone.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# ======================================================
# Style
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

sns.set_style("white")

# ======================================================
# Config
# ======================================================

OUTDIR = "fig3_final_validation_all_lr_bs"
os.makedirs(OUTDIR, exist_ok=True)

BAR_WIDTH = 0.22
GROUP_SPACING = 0.95

DISEASE_ORDER = ["CHD", "stroke", "istroke", "ICVD", "UA"]

DISEASE_XLABELS = [
    "Coronary\nHeart Disease",
    "Stroke",
    "Ischemic\nStroke",
    "ASCVD",
    "Unstable\nAngina"
]

MODEL_COLORS = {
    "CNN-1D": "#A3A9A6",
    "BiLSTM": "#6EB8D2",
    "Transformer": "#8B1E1E",
    "ResNet18": "#A3A9A6",
    "RetFound": "#6EB8D2",
    "ViT-Small (Frozen)": "#435385",
    "ViT-Small (Fine-tuned)": "#8B1E1E",
}

CURVE_DIRS = {
    "ECG": {
        "CNN-1D": "code_02_02_save/SingleModal_Run_update_v5/ecg_20251123_1316/curves",
        "BiLSTM": "code_02_02_save/SingleModal_Run_update_v5/ecg_20251123_1316/curves",
        "Transformer": "code_02_02_save/SingleModal_Run_update_v5/ecg_20251123_1316/curves",
    },
    "Fundus": {
        "ResNet18": "code_02_02_save/SingleModal_Run_update_v5/fundus_20251123_1314/curves",
        "RetFound": "code_02_02_save/SingleModal_Run_update_v5/fundus_20251123_1314/curves",
        "ViT-Small (Frozen)": "code_02_02_save/SingleModal_Run_update_v5/fundus_20251123_1314/curves",
        "ViT-Small (Fine-tuned)": "code_02_02_save/SingleModal_Run_update_v5/fundus_20251123_1314/curves",
    }
}

MODALITY_TAG = {
    "ECG": "ecg",
    "Fundus": "fundus"
}

MODEL_TAG = {
    "CNN-1D": "cnn1d",
    "BiLSTM": "bilstm",
    "Transformer": "transformer_h256_l3",
    "ResNet18": "resnet",
    "RetFound": "retfound",
    "ViT-Small (Frozen)": "vitsmallFREEZE",
    "ViT-Small (Fine-tuned)": "vitsmallFT",
}

MODEL_ORDER = {
    "ECG": ["CNN-1D", "BiLSTM", "Transformer"],
    "Fundus": [
        "ResNet18",
        "RetFound",
        "ViT-Small (Frozen)",
        "ViT-Small (Fine-tuned)"
    ]
}

# ======================================================
# Utils
# ======================================================

def extract_fold(filename):
    m = re.search(r"_fold(\d+)_", filename)
    return int(m.group(1)) if m else np.nan


def extract_lr(filename):
    m = re.search(r"_lr([^_]+)_bs", filename)
    return m.group(1) if m else np.nan


def extract_bs(filename):
    m = re.search(r"_bs([^_]+)_train_curve", filename)
    return m.group(1) if m else np.nan


def find_curve_files(modality, model, disease):
    curve_dir = CURVE_DIRS[modality][model]
    modality_tag = MODALITY_TAG[modality]
    model_tag = MODEL_TAG[model]

    pattern = os.path.join(
        curve_dir,
        f"{disease}_{modality_tag}_{model_tag}_fold*_lr*_bs*_train_curve.csv"
    )

    files = sorted(glob.glob(pattern))

    if len(files) == 0:
        print("[Missing]", modality, model, disease)
        print("Pattern:", pattern)

    return files


def get_best_valid_ci(modality, model, disease):

    files = find_curve_files(modality, model, disease)

    rows = []

    for f in files:
        try:
            df = pd.read_csv(f)
        except Exception as e:
            print("[Read error]", f, e)
            continue

        if "valid_ci" not in df.columns:
            print("[No valid_ci]", f)
            continue

        df["valid_ci"] = pd.to_numeric(df["valid_ci"], errors="coerce")
        df = df.dropna(subset=["valid_ci"])

        if len(df) == 0:
            continue

        idx = df["valid_ci"].idxmax()
        best = df.loc[idx]

        rows.append({
            "modality": modality,
            "model": model,
            "endpoint": disease,
            "fold": extract_fold(os.path.basename(f)),
            "lr": extract_lr(os.path.basename(f)),
            "bs": extract_bs(os.path.basename(f)),
            "best_epoch": int(best["epoch"]) if "epoch" in df.columns else np.nan,
            "best_valid_ci": float(best["valid_ci"]),
            "train_ci_at_best": float(best["train_ci"]) if "train_ci" in df.columns else np.nan,
            "train_loss_at_best": float(best["train_loss"]) if "train_loss" in df.columns else np.nan,
            "curve_file": f
        })

    if len(rows) == 0:
        return None

    tmp = pd.DataFrame(rows)
    return tmp.loc[tmp["best_valid_ci"].idxmax()].copy()


# ======================================================
# Collect best validation result
# ======================================================

all_results = []

for modality in ["ECG", "Fundus"]:
    for model in MODEL_ORDER[modality]:
        for disease in DISEASE_ORDER:

            best_row = get_best_valid_ci(
                modality=modality,
                model=model,
                disease=disease
            )

            if best_row is not None:
                all_results.append(best_row)

results = pd.DataFrame(all_results)

summary_csv = os.path.join(
    OUTDIR,
    "ECG_Fundus_best_validation_valid_ci_all_lr_bs.csv"
)

results.to_csv(summary_csv, index=False)

print("Saved:", summary_csv)
print(results)

# ======================================================
# Plot
# ======================================================

def add_value_labels(ax, bars):
    for bar in bars:
        h = bar.get_height()

        if np.isnan(h):
            continue

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.008,
            f"{h:.3f}",
            ha="center",
            va="bottom",
            fontsize=7.5,
            rotation=90
        )


def make_validation_barplot(modality):

    sub = results[results["modality"] == modality].copy()

    fig, ax = plt.subplots(figsize=(6.0, 4.0))

    x = np.arange(len(DISEASE_ORDER)) * GROUP_SPACING
    models = MODEL_ORDER[modality]
    width = BAR_WIDTH
    offsets = np.arange(len(models)) - (len(models) - 1) / 2

    for i, model in enumerate(models):

        vals = []

        for disease in DISEASE_ORDER:
            row = sub[
                (sub["endpoint"] == disease)
                & (sub["model"] == model)
            ]

            if len(row) == 0:
                vals.append(np.nan)
            else:
                vals.append(row.iloc[0]["best_valid_ci"])

        bars = ax.bar(
            x + offsets[i] * width,
            vals,
            width,
            color=MODEL_COLORS.get(model, "#A3A9A6"),
            edgecolor="black",
            linewidth=0.5,
            label=model,
            zorder=3
        )

        add_value_labels(ax, bars)

    ax.set_xticks(x)
    ax.set_xticklabels(DISEASE_XLABELS, fontsize=9)
    ax.set_ylabel("Validation C-index")

    ax.set_title(
        "ECG" if modality == "ECG" else "Fundus image",
        fontsize=14
    )

    ax.set_ylim(0.45, 0.90)

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3,
        zorder=0
    )

    ax.legend(
        frameon=False,
        fontsize=9,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.16),
        ncol=2
    )

    sns.despine(ax=ax)
    plt.tight_layout()

    out_pdf = os.path.join(
        OUTDIR,
        f"{modality}_validation_valid_ci_barplot_all_lr_bs.pdf"
    )

    out_png = os.path.join(
        OUTDIR,
        f"{modality}_validation_valid_ci_barplot_all_lr_bs.png"
    )

    plt.savefig(out_pdf, dpi=300, transparent=True, bbox_inches="tight")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    print("Saved:", out_pdf)
    print("Saved:", out_png)


for modality in ["ECG", "Fundus"]:
    make_validation_barplot(modality)

print("\nDone.")